In [ ]:
import os
import pandas as pd
import numpy as np

from gsm_benchmarker.results_analyser.bootstrap_result import BootstrapResult

from results_notebook_setup import RESULTS_ROOT, RESULTS_FOLDERS


In [ ]:
RESULTS_FOLDERS.keys()

In [ ]:
boot_path = (RESULTS_ROOT / RESULTS_FOLDERS['gsm']).parent / 'bootstrap'
os.listdir(boot_path)

In [ ]:
n_boot = 2000

bs1 = BootstrapResult(boot_path, n_boot=n_boot, effect='variant')

bs2 = BootstrapResult(boot_path, n_boot=n_boot, effect='number')
bs2.summary_df.rename(index={'sum_logs_c': 'gamma_c'}, level=1, inplace=True)
for v in bs2.full_results.values():
    v['estimates'].rename(columns={'sum_logs_c': 'gamma_c'}, inplace=True)


## GLMM 1 - variant effect

In [ ]:
bs1.summary_numbers  # bootstrap stats

In [ ]:
# summary of clean estimates
bs1.summary_boot

In [ ]:
# check how many models agree w.r.t. significance of the result; show the ones that don't
bs1.disagreements_check()

In [ ]:
# plot the distribution of estimates in the bootstrap for models which do not agree with single GLMM significance verdict
bs1.plot_nonagreeing_estimates('is_variant')

For all models - also the agreeing ones

In [ ]:
# quick scan for any other model where bootstrap mean diverges meaningfully from the original estimate
# (bias is <bootstrap mean estimate> minus <single GLMM estimate>
bs1.bias_check('is_variant')


In [ ]:
# CI width ratio (bootstrap CI width / original CI width) across all models, excluding the ones which did not converge for single GLMM
bs1.ci_width_check('is_variant')

In [ ]:
# bootstrap estimate distribution skew check across all models
bs1.skew_check('is_variant')

## GLMM2

In [ ]:
bs2.summary_numbers.xs('is_variant', level=1)  # shared for both variables


### a) number effect

In [ ]:
bs2.disagreements_check('gamma_c')

In [ ]:
bs2.plot_nonagreeing_estimates('gamma_c')

In [ ]:
bs2.bias_check('gamma_c')

In [ ]:
bs2.ci_width_check('gamma_c')

In [ ]:
bs2.skew_check('gamma_c')

### b) number-effect-corrected variant effect

In [ ]:
bs2.disagreements_check('is_variant')

In [ ]:
bs2.plot_nonagreeing_estimates('is_variant')

In [ ]:
bs2.bias_check('is_variant')


In [ ]:

bs2.ci_width_check('is_variant')

In [ ]:

bs2.skew_check('is_variant')

---
# Combined analysis for all prompt formats

In [ ]:
n_boot = 2000

boots = {
    'GLMM1': {},
    'GLMM2': {},
}

for prompt_name, prompt_folder in RESULTS_FOLDERS.items():
    boot_path = (RESULTS_ROOT / prompt_folder).parent / 'bootstrap'

    bs1 = BootstrapResult(boot_path, n_boot=n_boot, effect='variant')

    bs2 = BootstrapResult(boot_path, n_boot=n_boot, effect='number')
    bs2.summary_df.rename(index={'sum_logs_c': 'gamma_c'}, level=1, inplace=True)
    for v in bs2.full_results.values():
        v['estimates'].rename(columns={'sum_logs_c': 'gamma_c'}, inplace=True)

    boots['GLMM1'][prompt_name] = bs1
    boots['GLMM2'][prompt_name] = bs2



In [ ]:
bs2.summary_df.xs('is_variant', level=1)

In [ ]:
import pandas as pd

def combine_prompt_boots(glmm_key, effect):
    return pd.concat({key: value.summary_df.xs(effect, level=1) for key, value in boots[glmm_key].items()}, axis=0, names=['prompt', 'model'])

boots_combined = {
    'GLMM1-is_variant': combine_prompt_boots('GLMM1', 'is_variant'),
    'GLMM2-is_variant': combine_prompt_boots('GLMM2', 'is_variant'),
    'GLMM2-gamma_c': combine_prompt_boots('GLMM2', 'gamma_c')
}

boots_combined['GLMM1-is_variant']

In [ ]:
original_significance = boots_combined['GLMM1-is_variant'].xs('gsm', level='prompt').single_significant
significant_models = original_significance[original_significance].index.tolist()
significant_models

In [ ]:
for combo_key, combo_summary in boots_combined.items():
    for prompt_name in combo_summary.index.get_level_values('prompt').unique():
        if 'GLMM1' in combo_key and prompt_name == 'gsm':
            continue
        for model_name in combo_summary.xs(prompt_name, level='prompt').index:
            if model_name not in significant_models:
                combo_summary.drop(index=(prompt_name, model_name), inplace=True)


In [ ]:
combo_summary

In [ ]:
def make_summary(c):
    prompt_group = c.reset_index().groupby('prompt')

    n_models = prompt_group.size()
    n_agreement = prompt_group.agreement.sum()

    convergent_prompt_group = c[~c.single_nonconvergent].reset_index().groupby('prompt')
    width_ratio_mean = convergent_prompt_group.width_ratio.mean()
    width_ratio_sd = convergent_prompt_group.width_ratio.std()

    bias = convergent_prompt_group.bias

    s = pd.concat({
        'N models': n_models,
        'Agreement': pd.Series([f"{a}/{n}" for a, n in zip(n_agreement, n_models)], index=n_agreement.index),
        # 'N singular': prompt_group.single_singular.sum(),
        'N non-convergent': prompt_group.single_nonconvergent.sum(),
        'CI width ratio: mean ± SD': pd.Series([f"{mean:.3f} ± {sd:.3f}" for mean, sd in zip(width_ratio_mean, width_ratio_sd)], index=width_ratio_mean.index),
        'Max|bias|': pd.Series(np.maximum(bias.max(), -bias.min()), index=width_ratio_mean.index),
    }, axis=1).loc[RESULTS_FOLDERS.keys()]
    return s

make_summary(boots_combined['GLMM1-is_variant'])

In [ ]:
make_summary(boots_combined['GLMM2-is_variant'])

In [ ]:
make_summary(boots_combined['GLMM2-gamma_c'])

In [ ]:
boots_combined_single_df = pd.concat(boots_combined, names=['test'])
boots_combined_single_df

In [ ]:
boots_combined_single_df[~boots_combined_single_df.agreement][['single_significant', 'boot_significant', 'single_nonconvergent']].sort_values(['single_nonconvergent']).sort_index(level='prompt')

In [ ]:
boots_combined_single_df[boots_combined_single_df.single_nonconvergent][['agreement', 'single_significant', 'boot_significant']]